# Tool Calls

In [ ]:
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    current_step: str



@tool
def calculate_cost(minutes: int) -> str:
    """Calculates the hosting cost for running a cloud server for a given number of minutes."""
    rate_per_minute = 0.50 / 60
    total = minutes * rate_per_minute
    return f"The total cost for running the server for {minutes} minutes is ${total:.2f} USD."


tool_node = ToolNode([calculate_cost])


def initiating_operation(state: AgentState):
    """A Standard python code that runs before LLM"""

    # A stricter system prompt to force Llama 3 to use the tool
    system_prompt = (
        "You are a helpful cloud cost assistant. "
        "If the user asks for an estimate or cost regarding a duration of time, "
        "you MUST use the 'calculate_cost' tool to provide the answer."
    )

    # Prepend the system prompt to the existing user messages in the state
    return {
        "current_step": "Initialization Complete",
        "messages": [{"role": "system", "content": system_prompt}] + state["messages"],
    }


def router_node(state: AgentState):
    """Determine whether to navigate to Tool Node or Finish Execution"""
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "execute_tools"

    return END
#from langchain_ollama import ChatOllama
from langgraph.config import get_stream_writer
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

"""
llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0,
    base_url="http://localhost:11434" # Default local Ollama port
)
"""
llm = ChatOpenAI(
    model=os.getenv("AI_MODEL"),
    temperature=0)

llm.bind_tools([calculate_cost], tool_choice="required")

def llm_assistant_node(state: AgentState):
    """Node that triggers LLM to either stream a response or call a tool"""
    writer = get_stream_writer()
    writer({"info": "Llama 3 is processing your query."})
    response = llm.invoke(state["messages"])
    return {"current_step": "Assistant Processed", "messages": [response]}

from IPython.display import display, Image

builder = StateGraph(AgentState)

builder.add_node("initiating_operation", initiating_operation)
builder.add_node("llm_assistant_node", llm_assistant_node)
builder.add_node("execute_tools", tool_node)

builder.add_edge(START, "initiating_operation")
builder.add_edge("initiating_operation", "llm_assistant_node")
builder.add_edge("execute_tools", "llm_assistant_node")

builder.add_conditional_edges(
    "llm_assistant_node",
    router_node, # No quotes around the function object!
    {
        "execute_tools": "execute_tools",
        END: END
    }
)

graph = builder.compile()


display(Image(graph.get_graph().draw_mermaid_png()))



async def operation(user_message):
    # Your baseline state inputs
    inputs = {"current_step": "Started", "messages": [{"role": "user", "content": user_message}]}
    print("Agent is thinking ...")

    # Using version="v2" guarantees the flat dictionary structure with the "event" key
    async for event in graph.astream_events(inputs, version="v2"):
        event_type = event["event"]
        name = event["name"]

        # 1. Capture Normal Node Lifecycles
        if event_type == "on_node_start":
            print(f"\n🟩 [Node Started] -> {name}")
        elif event_type == "on_node_end":
            print(f"🟥 [Node Finished] -> {name}")

        # 2. Capture Tool Invocations Natively
        elif event_type == "on_tool_start":
            print(f"🛠️  [Tool Active]: '{name}' running with arguments: {event['data'].get('input')}")
        elif event_type == "on_tool_end":
            print(f"📦 [Tool Output]: Result received -> {event['data'].get('output').content}")

        # 3. Capture Live Text Tokens Streamed from Llama 3
        elif event_type == "on_chat_model_stream":
            chunk = event["data"].get("chunk")
            if chunk and chunk.content:
                print(chunk.content, end="", flush=True)

        # 4. Capture Custom get_stream_writer() Pings
        elif event_type == "on_custom_event":
            data = event["data"]
            # Fallbacks to extract text safely depending on how you formatted your dictionary keys
            msg = data.get("info") or data.get("message") or data.get("progress") or data
            print(f"\n⚡ [Custom Status Message]: {msg}")

    print("\n....")

import asyncio
async def chat():
    for _ in range(3):
        user_message = input("\nYou: ")
        await operation(user_message)

await chat()